In [ ]:
"""
Boxplot rmsd for DA-experiments comparison on GPP (sites & grid cells) & SIF (grid-cells)
@V. Tartaglione/F. Maignan
"""
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
from datetime import datetime
import cftime
import xarray as xr
import seaborn as sns
from IPython import embed
import sys

In [ ]:
def calculate_rmsd(obs, model):
    """Calculate Root Mean Square Deviation (RMSD) between observations and model data."""
    # Convert inputs to numpy arrays if they aren't already
    obs_array = np.array(obs, dtype=float)
    model_array = np.array(model, dtype=float)
    
    # Check for empty arrays
    if obs_array.size == 0 or model_array.size == 0:
        # This warning is suppressed as it can be noisy during normal operation
        # print("Warning: Empty array detected in RMSD calculation")
        return np.nan
        
    # Calculate RMSD
    return np.sqrt(np.nanmean((obs_array - model_array)**2))

In [ ]:
def replace_extreme_values(data, threshold=1e10):
    """Replace values larger than the threshold with NaN."""
    return np.where(np.abs(data) > threshold, np.nan, data)

In [ ]:
def convert_to_datetime(cftime_obj):
    if isinstance(cftime_obj, cftime.datetime):
        return datetime(cftime_obj.year, cftime_obj.month, cftime_obj.day, cftime_obj.hour, cftime_obj.minute, cftime_obj.second)
    return None

In [ ]:
def read_pixel_numbers_from_csv_simple(csv_path):
    """
    Reads pixel numbers from a CSV and returns them as a list of 
    zero-padded strings (e.g., '01', '02', '21').
    """
    if not os.path.exists(csv_path):
        print(f"ERROR: Evaluation pixel file not found: {csv_path}")
        return []
    try:
        df = pd.read_csv(csv_path)
        pixel_col = next((col for col in ['pixel_number', 'pixel', 'px'] if col in df.columns), None)
        if not pixel_col:
            print(f"ERROR: No pixel column found in {csv_path}")
            return []
        
        # 1. Convert to numeric, drop invalid entries, convert to integer
        # 2. Apply a formatting function to each integer
        # 3. Convert the result to a list of strings
        return df[pixel_col].dropna().astype(int).apply(lambda x: f"{x:02d}").tolist()

    except Exception as e:
        print(f"ERROR: Could not read pixel CSV file: {e}")
        return []

In [ ]:
def align_data_to_common_range(data_dict):
    """More robust time alignment using pandas merge."""
    series_list = []
    for key, (data, time_index) in data_dict.items():
        s = pd.Series(data, index=pd.to_datetime(time_index), name=key)
        # Handle potential duplicate timestamps
        if not s.index.is_unique:
            s = s.groupby(s.index).mean()
        series_list.append(s)
    
    # Inner join finds the intersection of all time series
    df_merged = pd.concat(series_list, axis=1, join='inner').dropna()

    if df_merged.empty:
        return None, None, None

    start_date = df_merged.index.min()
    end_date = df_merged.index.max()
    
    # Return a dictionary of aligned data arrays and the date range
    aligned_data = {col: (df_merged[col].values, df_merged.index) for col in df_merged.columns}
    return aligned_data, start_date, end_date

In [ ]:
def load_gpp_data(prior_file, post_file, fluxcom_file, fluxsat_file):
    # Load prior GPP data
    ds1 = xr.open_dataset(prior_file)
    GPPt = ds1['data_site0_var1'][1, :].values
    site_id = ds1['site_id'][0].values

    # Load post GPP data
    ds2 = xr.open_dataset(post_file)
    GPPt_post = ds2['data_site0_var0'][1, :].values

    # Extract time information for both prior and post
    time_units_gpp = ds1['data_site0_var0'].attrs['time_units']
    time_base_gpp = ds1['data_site0_var0'].attrs['time_offset']
    time_step_gpp = ds1['data_site0_var0'].attrs['time_step']
    time_prior = np.array([time_base_gpp + i * time_step_gpp for i in range(GPPt.shape[0])])
    time_prior = nc.num2date(time_prior, time_units_gpp)
    time_prior_datetime = np.array([convert_to_datetime(t) for t in time_prior])
    
    time_units_gpp_post = ds2['data_site0_var0'].attrs['time_units']
    time_base_gpp_post = ds2['data_site0_var0'].attrs['time_offset']
    time_step_gpp_post = ds2['data_site0_var0'].attrs['time_step']
    time_post = np.array([time_base_gpp_post + i * time_step_gpp_post for i in range(GPPt_post.shape[0])])
    time_post = nc.num2date(time_post, time_units_gpp_post)
    time_post_datetime = np.array([convert_to_datetime(t) for t in time_post])
    
    ################
    ###############
    # Process FLUXCOM file
    fluxcom = xr.open_dataset(fluxcom_file)
    GPP_FLUXCOM = fluxcom['GPP'].values.flatten()
    time_fluxcom = pd.to_datetime(fluxcom['time'].values)

    # Process FLUXSAT file with correction for reference date
    fluxsat = xr.open_dataset(fluxsat_file)
    GPP_FLUXSAT = fluxsat['GPP'].values.flatten()

    # Convert to datetime objects first
    fluxsat_datetime = pd.to_datetime(fluxsat['time'].values)

    # Calculate the offset: 2001-01-01 to 2020-01-01 is 366 days (2000 was leap year)
    offset = pd.Timedelta(days=366)

    # Subtract the offset to adjust the reference date
    time_fluxsat = fluxsat_datetime - offset

    # Get length of time series and dates
    fluxcom_days_count = len(time_fluxcom)
    fluxsat_days_count = len(time_fluxsat)

    # Get start and end dates
    fluxcom_start_date = time_fluxcom[0]
    fluxcom_end_date = time_fluxcom[-1]
    fluxsat_start_date = time_fluxsat[0]
    fluxsat_end_date = time_fluxsat[-1]
    ###########
    ###########
    
    # Prepare data dictionary for alignment
    data_dict = {
        'prior': (GPPt, time_prior_datetime),
        'post': (GPPt_post, time_post_datetime),
        'fluxcom': (GPP_FLUXCOM, time_fluxcom),
        'fluxsat': (GPP_FLUXSAT, time_fluxsat)
    }
    
    # Apply the align_data_to_common_range function
    aligned_data, start_date, end_date = align_data_to_common_range(data_dict)
    
    # Create the final gpp_data dictionary
    gpp_data = {
        'prior': aligned_data['prior'][0],
        'post': aligned_data['post'][0],
        'fluxcom': aligned_data['fluxcom'][0],
        'fluxsat': aligned_data['fluxsat'][0],
        'time': aligned_data['post'][1]  # Using post time as the common time
    }
    
    return gpp_data

In [ ]:
def extract_gpp_from_file(file_path, site_id, data_type):
    try:
        with nc.Dataset(file_path) as ds:
            # GPP is var1, SIF is var0. This function is for GPP.
            var_name = 'data_site0_var1'
            if var_name not in ds.variables: return None
            gpp_obs = replace_extreme_values(ds.variables[var_name][0, :].data)
            gpp_sim = replace_extreme_values(ds.variables[var_name][1, :].data)
            if np.all(np.isnan(gpp_obs)) or np.all(np.isnan(gpp_sim)): return None
            return {'obs': gpp_obs, 'sim': gpp_sim}
    except Exception:
        return None

In [ ]:
def process_fluxnet_gpp(prior_dir, post_sif_dir, post_gpp_dir, post_sif_gpp_dir, post_sif_gpp_bacour_dir, site_filter_list):
    fluxnet_gpp_rmsd = {'prior': [], 'post_sif': [], 'post_gpp': [], 'post_sif_gpp': [], 'post_sif_gpp_bacour': []}
    raw_data = {'obs': [], 'prior': [], 'post_sif': [], 'post_gpp': [], 'post_sif_gpp': [], 'post_sif_gpp_bacour': []}
    
    print(f"\nProcessing FLUXNET sites. Applying filter for {len(site_filter_list)} sites.")
    
    for site_id in site_filter_list: 
        try:
            files = {
                'prior': get_most_recent_file(os.path.join(prior_dir, site_id, "output_*.nc")),
                'post_sif': get_most_recent_file(os.path.join(post_sif_dir, site_id, "output_*.nc")),
                'post_gpp': get_most_recent_file(os.path.join(post_gpp_dir, site_id, "output_*.nc")),
                'post_sif_gpp': get_most_recent_file(os.path.join(post_sif_gpp_dir, site_id, "output_*.nc")),
                'post_sif_gpp_bacour': get_most_recent_file(os.path.join(post_sif_gpp_bacour_dir, site_id, "output_*.nc")),
            }
            if not all(files.values()): continue

            data = {key: extract_gpp_from_file(f, site_id, key) for key, f in files.items()}
            if not all(data.values()): continue

            gpp_obs, sim_data = data['prior']['obs'], {key: val['sim'] for key, val in data.items()}
            valid_mask = ~np.isnan(gpp_obs)
            for sim in sim_data.values(): valid_mask &= ~np.isnan(sim)

            if np.sum(valid_mask) > 0:
                obs_valid = gpp_obs[valid_mask]
                raw_data['obs'].extend(obs_valid)
                for key, sim_valid in sim_data.items():
                    sim_valid = sim_valid[valid_mask]
                    fluxnet_gpp_rmsd[key].append(calculate_rmsd(obs_valid, sim_valid))
                    raw_data[key].extend(sim_valid)
        except Exception:
            continue
    return fluxnet_gpp_rmsd, raw_data

In [ ]:
def load_eval_sif_pixels(post_sif_dir, post_gpp_dir, post_sif_gpp_dir, post_sif_gpp_bacour_dir, prior_dir, pixel_filter_list):
    sif_eval = {'obs': [], 'prior': [], 'post_sif': [], 'post_gpp': [], 'post_sif_gpp': [], 'post_sif_gpp_bacour': []}
    sif_eval_rmsd = {'prior': [], 'post_sif': [], 'post_gpp': [], 'post_sif_gpp': [], 'post_sif_gpp_bacour': []}
    
    print(f"\nProcessing SIF pixels. Applying filter for {len(pixel_filter_list)} pixels.")

    for px_num in pixel_filter_list:
        px = str(px_num)
        try:
            files = {
                'prior': get_most_recent_file(os.path.join(prior_dir, f"PX{px}", "output_*.nc")),
                'post_sif': get_most_recent_file(os.path.join(post_sif_dir, f"PX{px}", "output_*.nc")),
                'post_gpp': get_most_recent_file(os.path.join(post_gpp_dir, f"PX{px}", "output_*.nc")),
                'post_sif_gpp': get_most_recent_file(os.path.join(post_sif_gpp_dir, f"PX{px}", "output_*.nc")),
                'post_sif_gpp_bacour': get_most_recent_file(os.path.join(post_sif_gpp_bacour_dir, f"PX{px}", "output_*.nc")),
            }
            if not all(files.values()): continue
            
            sif_var_name = 'data_site0_var0'
            with nc.Dataset(files['prior']) as ds:
                obs_prior = replace_extreme_values(ds[sif_var_name][0, :].data)
                sim_data = {'prior': replace_extreme_values(ds[sif_var_name][1, :].data)}
            
            for key, path in files.items():
                if key != 'prior':
                    with nc.Dataset(path) as ds: sim_data[key] = replace_extreme_values(ds[sif_var_name][1, :].data)
            
            valid_mask = ~np.isnan(obs_prior)
            for sim in sim_data.values(): valid_mask &= ~np.isnan(sim)
            
            if np.sum(valid_mask) > 0:
                obs_valid = obs_prior[valid_mask]
                sif_eval['obs'].extend(obs_valid)
                for key, sim in sim_data.items():
                    sim_valid = sim[valid_mask]
                    sif_eval[key].extend(sim_valid)
                    sif_eval_rmsd[key].append(calculate_rmsd(obs_valid, sim_valid))
        except Exception:
            continue
    return sif_eval, sif_eval_rmsd

In [ ]:
def plot_rmsd_boxplots(gpp_rmsd,
                           fluxnet_gpp_rmsd,
                           sif_eval_rmsd,
                           sif_cell_count,
                           gpp_site_count,
                           gpp_cell_count):
    #sns.set_style("darkgrid")
    plt.rcParams.update({'font.size': 12})

    # IBM ColorBlind Safe palette
    ibm_colors = [
        '#648FFF', '#785EF0', '#DC267F', '#FE6100', '#FFB000',
        '#009E73', '#56B4E9', '#E69F00', '#CC79A7', '#005AB5',
        '#F0E442', '#E6AB02', '#00A087', '#882255', '#AA4499'
    ]
    
    # Color definitions
    colors = {
        'Prior': sns.color_palette("colorblind")[1],
        'Post. SIF': sns.color_palette("colorblind")[2],
        'Post. GPP': sns.color_palette("colorblind")[-1],
        'Post. SIF-GPP\nLeverne': sns.color_palette("colorblind")[4],
        'Post. SIF-GPP\nBacour': ibm_colors[13],
        'Prior vs FLUXCOM-X': sns.color_palette("colorblind")[1],
        'Prior vs FLUXSAT': sns.color_palette("colorblind")[1],
        'Post. SIF vs FLUXCOM-X': sns.color_palette("colorblind")[2],
        'Post. SIF vs FLUXSAT': sns.color_palette("colorblind")[2],
        'Post. GPP vs FLUXCOM-X': sns.color_palette("colorblind")[-1],
        'Post. GPP vs FLUXSAT': sns.color_palette("colorblind")[-1],
        'Post. SIF-GPP Leverne vs FLUXCOM-X': sns.color_palette("colorblind")[4],
        'Post. SIF-GPP Leverne vs FLUXSAT': sns.color_palette("colorblind")[4],
        'Post. SIF-GPP Bacour vs FLUXCOM-X': ibm_colors[13],
        'Post. SIF-GPP Bacour vs FLUXSAT': ibm_colors[13]
    }
    
    # Create figure with three subplots
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 5))
    
    def add_stats_textbox(ax, data, concat_val, pos, color, format_str='.3f', x_pos=1.02):
        """Helper function to add statistics textbox with adjustable x position"""
        if isinstance(data, (list, np.ndarray)) and len(data) > 0:
            mean_val = np.nanmean(data)
            median_val = np.nanmedian(data)
            std_val = np.nanstd(data)
            
            stats_text = (f'Mean: {mean_val:{format_str}}\n'
                         f'Median: {median_val:{format_str}}\n'
                         f'Std: {std_val:{format_str}}\n'
                         f'CRMSD: {concat_val:{format_str}}')
            
            props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=color)
            ax.text(x_pos, pos, stats_text, transform=ax.transAxes, fontsize=10,
                   verticalalignment='center', bbox=props)
                   
    # Plot 1: SIF RMSD
    positions_sif = [4.5, 3.5, 2.5, 1.5, 0.5]  # From top to bottom
    labels_sif = ['Prior', 'Post. SIF', 'Post. GPP', "Post. SIF-GPP\nLeverne", "Post. SIF-GPP\nBacour"]
    data_sif = [
        sif_eval_rmsd['prior'],
        sif_eval_rmsd['post_sif'],
        sif_eval_rmsd['post_gpp'],
        sif_eval_rmsd['post_sif_gpp'],
        sif_eval_rmsd['post_sif_gpp_bacour']
    ]

    means = {name: np.mean(values) for name, values in sif_eval_rmsd.items()}
    
    colors_sif = [colors[label] for label in labels_sif]
    
    # Dictionary to map display labels to dictionary keys
    key_mapping = {
        'Prior': 'prior',
        'Post. SIF': 'post_sif',
        'Post. GPP': 'post_gpp',
        'Post. SIF-GPP\nLeverne': 'post_sif_gpp',
        'Post. SIF-GPP\nBacour': 'post_sif_gpp_bacour'
    }
    
    bp1 = ax1.boxplot(data_sif, positions=positions_sif, widths=0.4, 
                      patch_artist=True, vert=False, showmeans=True, meanline=True, 
                      meanprops={'color': 'black', 'linestyle': 'solid'},  # Mean line style
                      medianprops={'visible': False})
    
    # Color SIF boxes and add statistics
    for i, (patch, color, data) in enumerate(zip(bp1['boxes'], colors_sif, data_sif)):
        patch.set_facecolor(color)
        # Use the key mapping to get the correct dictionary key
        dict_key = key_mapping[labels_sif[i]]
        """add_stats_textbox(ax1, data, concat_rmsd_sif_eval[dict_key],
                         pos=(4-i)/4.5, color=color)"""

    ax1.set_facecolor('white')
    ax1.spines['bottom'].set_color('black')
    ax1.spines['left'].set_color('black')
    
    plt.setp(bp1['medians'], color='black')
    plt.setp(bp1['whiskers'], color='black')
    plt.setp(bp1['caps'], color='black')
    
    # Add y-axis labels for SIF
    ax1.set_yticks(positions_sif)
    ax1.set_yticklabels(labels_sif)

    # Plot 2: FLUXNET GPP RMSD
    positions_fluxnet = [4.5, 3.5, 2.5, 1.5, 0.5]  # From top to bottom
    labels_fluxnet = ['Prior', 'Post. SIF', 'Post. GPP', 'Post. SIF-GPP\nLeverne', 'Post. SIF-GPP\nBacour']
    data_fluxnet = [
        fluxnet_gpp_rmsd['prior'],
        fluxnet_gpp_rmsd['post_sif'],
        fluxnet_gpp_rmsd['post_gpp'],
        fluxnet_gpp_rmsd['post_sif_gpp'],
        fluxnet_gpp_rmsd['post_sif_gpp_bacour']
    ]
    
    means = {name: np.mean(values) for name, values in fluxnet_gpp_rmsd.items()}

    colors_fluxnet = [colors[label] for label in labels_fluxnet]
    
    bp2 = ax2.boxplot(data_fluxnet, positions=positions_fluxnet, widths=0.4,
                      patch_artist=True, vert=False, showmeans=True, meanline=True, 
                      meanprops={'color': 'black', 'linestyle': 'solid'},  # Mean line style
                      medianprops={'visible': False})
    
    # Color FLUXNET boxes and add statistics
    for i, (patch, color, data) in enumerate(zip(bp2['boxes'], colors_fluxnet, data_fluxnet)):
        patch.set_facecolor(color)
        dict_key = key_mapping[labels_fluxnet[i]]

    ax2.set_facecolor('white')
    ax2.spines['bottom'].set_color('black')
    ax2.spines['left'].set_color('black')
    
    plt.setp(bp2['medians'], color='black')
    plt.setp(bp2['whiskers'], color='black')
    plt.setp(bp2['caps'], color='black')
    
    # Add y-axis labels for FLUXNET
    ax2.set_yticks(positions_fluxnet)
    ax2.set_yticklabels('')
    
    # Plot 3: GPP RMSD
    positions_gpp = [
        8.3, 7.5,      # Prior FLUXCOM & FLUXSAT
        6.0, 5.2,      # Post. SIF FLUXCOM & FLUXSAT
        3.7, 2.9,      # Post. GPP FLUXCOM & FLUXSAT
        1.4, 0.6,      # Post. SIF-GPP FLUXCOM & FLUXSAT
       -0.9,-1.7       # Post. SIF-GPP Bacour FLUXCOM & FLUXSAT
    ]
    
    
    labels_gpp = [
        'Prior vs FLUXCOM-X',
        'Prior vs FLUXSAT',
        'Post. SIF vs FLUXCOM-X',
        'Post. SIF vs FLUXSAT',
        'Post. GPP vs FLUXCOM-X',
        'Post. GPP vs FLUXSAT',
        'Post. SIF-GPP Leverne vs FLUXCOM-X',
        'Post. SIF-GPP Leverne vs FLUXSAT',
        'Post. SIF-GPP Bacour vs FLUXCOM-X',
        'Post. SIF-GPP Bacour vs FLUXSAT'
    ]
    
    # Dictionary to map GPP labels to dictionary keys
    gpp_key_mapping = {
        'Prior vs FLUXCOM-X': 'prior_fluxcom',
        'Prior vs FLUXSAT': 'prior_fluxsat',
        'Post. SIF vs FLUXCOM-X': 'post_sif_fluxcom',
        'Post. SIF vs FLUXSAT': 'post_sif_fluxsat',
        'Post. GPP vs FLUXCOM-X': 'post_gpp_fluxcom',
        'Post. GPP vs FLUXSAT': 'post_gpp_fluxsat',
        'Post. SIF-GPP Leverne vs FLUXCOM-X': 'post_sif_gpp_fluxcom',
        'Post. SIF-GPP Leverne vs FLUXSAT': 'post_sif_gpp_fluxsat',
        'Post. SIF-GPP Bacour vs FLUXCOM-X': 'post_sif_gpp_bacour_fluxcom',
        'Post. SIF-GPP Bacour vs FLUXSAT': 'post_sif_gpp_bacour_fluxsat'
    }
    
    data_gpp = [
        gpp_rmsd['prior_fluxcom'],
        gpp_rmsd['prior_fluxsat'],
        gpp_rmsd['post_sif_fluxcom'],
        gpp_rmsd['post_sif_fluxsat'],
        gpp_rmsd['post_gpp_fluxcom'],
        gpp_rmsd['post_gpp_fluxsat'],
        gpp_rmsd['post_sif_gpp_fluxcom'],
        gpp_rmsd['post_sif_gpp_fluxsat'],
        gpp_rmsd['post_sif_gpp_bacour_fluxcom'],
        gpp_rmsd['post_sif_gpp_bacour_fluxsat']
    ]
    
    means = {name: np.mean(values) for name, values in gpp_rmsd.items()}

    means = {name: np.mean(values) for name, values in gpp_rmsd.items()}
    
    colors_gpp = [colors[label] for label in labels_gpp]
    
    bp3 = ax3.boxplot(data_gpp, positions=positions_gpp, widths=0.5,
                      patch_artist=True, vert=False, showmeans=True, meanline=True, 
                      meanprops={'color': 'black', 'linestyle': 'solid'},  # Mean line style
                      medianprops={'visible': False})
    group_positions = [0.90, 0.7, 0.5, 0.3, 0.1]
    
    # Process GPP boxes in pairs
    for i in range(0, len(data_gpp), 2):
        # FLUXCOM data
        patch_fluxcom = bp3['boxes'][i]
        color_fluxcom = colors_gpp[i]
        data_fluxcom = data_gpp[i]
        dict_key_fluxcom = gpp_key_mapping[labels_gpp[i]]
        
        # FLUXSAT data
        patch_fluxsat = bp3['boxes'][i+1]
        color_fluxsat = colors_gpp[i+1]
        data_fluxsat = data_gpp[i+1]
        dict_key_fluxsat = gpp_key_mapping[labels_gpp[i+1]]
        
        # Set colors and transparency
        patch_fluxcom.set_facecolor(color_fluxcom)
        patch_fluxsat.set_facecolor(color_fluxsat)
        patch_fluxsat.set_alpha(0.6)
   
    ax3.set_facecolor('white')
    ax3.spines['bottom'].set_color('black')
    ax3.spines['left'].set_color('black')

    plt.setp(bp3['medians'], color='black')
    plt.setp(bp3['whiskers'], color='black')
    plt.setp(bp3['caps'], color='black')

    for i in range(len(data_gpp)):
        ls = 'dotted' if i % 2 == 0 else 'solid'  # FLUXCOM dotted, FLUXSAT solid
    
        # Mean line
        bp3['means'][i].set_linestyle(ls)
    
        # Whiskers (2 per box)
        bp3['whiskers'][2*i].set_linestyle(ls)
        bp3['whiskers'][2*i+1].set_linestyle(ls)
    
        # Caps (2 per box)
        bp3['caps'][2*i].set_linestyle(ls)
        bp3['caps'][2*i+1].set_linestyle(ls)
    
        # Box border
        bp3['boxes'][i].set_linestyle(ls)    
        # Add y-axis labels for GPP
        ax3.set_yticks(positions_gpp)
        ax3.set_yticklabels('')
 
    # Add legend for FLUXCOM vs FLUXSAT
    fluxcom_patch = plt.Rectangle((0, 0), 1, 1, fc='gray', edgecolor='black', linestyle='dotted')
    fluxsat_patch = plt.Rectangle((0, 0), 1, 1, fc='gray', alpha=0.6, edgecolor='black')
    ax3.legend([fluxcom_patch, fluxsat_patch], ['FLUXCOM', 'FLUXSAT'], 
              loc='lower right', bbox_to_anchor=(1.15, 0.000), prop={'size': 12}, labelspacing=0.5)

    # Customize plots
    for ax, title in zip([ax1, ax2, ax3], 
                        ['(a) SIF\n(20 independent grid cells)',
                         '(b) GPP\n(11 independent EC sites)',
                         '(c) GPP\n(20 independent grid cells)']):
        ax.set_title(title, pad=20, fontsize=14)
    
    # Set specific y-axis limits
    ax2.set_ylim(0, 5)
    ax3.set_ylim(-2.5, 9)

    # Find common x-axis limits for GPP plots
    gpp_data = data_gpp + data_fluxnet  # Combine all GPP data
    all_gpp_values = []
    for d in gpp_data:
        if isinstance(d, (list, np.ndarray)) and len(d) > 0:
            all_gpp_values.extend(d)
    
    # Set x-axis limits and ticks
    if len(all_gpp_values) > 0:
        max_rmsd_gpp = np.nanpercentile(all_gpp_values, 100)
        max_rmsd_gpp = np.ceil(max_rmsd_gpp)
        tick_spacing = max_rmsd_gpp / 5 if max_rmsd_gpp > 5 else 1
        ticks_gpp = np.arange(0, max_rmsd_gpp + tick_spacing, tick_spacing)
        
        # Apply to both GPP plots
        ax3.set_xlim(0, max_rmsd_gpp)
        ax2.set_xlim(0, max_rmsd_gpp)
        ax3.set_xticks(ticks_gpp)
        ax2.set_xticks(ticks_gpp)
        ax3.xaxis.set_major_formatter(plt.FormatStrFormatter('%.1f'))
        ax2.xaxis.set_major_formatter(plt.FormatStrFormatter('%.1f'))
    
    # Set SIF plot x-axis limits
    all_sif_values = []
    for d in data_sif:
        if isinstance(d, (list, np.ndarray)) and len(d) > 0:
            all_sif_values.extend(d)
            
    if len(all_sif_values) > 0:
        max_rmsd_sif = np.nanpercentile(all_sif_values, 99)
        ax1.set_xlim(0, max_rmsd_sif * 1.1)
    
    # Set x-label
    for ax in [ax1, ax2, ax3]:
        ax.set_xlabel('RMSD (gC$\cdot$m$^{-2}\cdot$d$^{-1}$)' if ax != ax1 else 'RMSD (W$\cdot$m$^{-2}\cdot\mu$m$^{-1}\cdot$sr$^{-1})$', fontsize=12)
    
    # Adjust layout to accommodate statistics textboxes
    plt.tight_layout()
    fig.subplots_adjust(right=0.95)  # Make room for the textboxes
    
    plt.show(block=False)
    #plt.savefig('./PNG/Figure1.png', 
    #            dpi=300, bbox_inches='tight')


In [ ]:
def get_pixel_number(filepath):
    basename = os.path.basename(filepath)
    match = re.search(r'PX(\d+)', basename, re.IGNORECASE)
    return match.group(1) if match else None

In [ ]:
def get_most_recent_file(file_pattern):
    files = glob.glob(file_pattern)
    return max(files, key=os.path.getmtime) if files else None

In [ ]:
# ==========================================================
# --- Main Execution Block ---
# ==========================================================
if __name__ == "__main__":

    # --- 1. EVALUATION GC ---
    EVAL_PIXELS_FILE = "/home/users/vtartagl/Assim_SIF-GPP/assim-eval_distribution/GC/evaluation_pixels_pft07_ampl_3.csv"
    EVALUATION_PIXELS = read_pixel_numbers_from_csv_simple(EVAL_PIXELS_FILE)
    EVALUATION_SITES = ['FI-Ken', 'FI-Sod', 'SE-Svb', 'CA-NS5', 'CA-NS6', 'CA-NS7', 'US-Syv',
                'CA-SF1','CH-Dav','IT-Ren','US-Prr']
    print(f"Loaded {len(EVALUATION_PIXELS)} evaluation pixels and {len(EVALUATION_SITES)} evaluation sites.")

    # --- 2. PATHS ---
    # SIF GC
    eval_post_sif_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/SIF_POST_SIF/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Lev"
    eval_post_gpp_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/SIF_POST_GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Lev"
    eval_post_sif_gpp_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/SIF_POST_SIF-GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Lev"
    eval_post_sif_gpp_bacour_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/SIF_POST_SIF-GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Bac"
    prior_sif_path = "/home/surface9/vtartagl/Pixel_homogenes/Simus/PRIOR_SIF_2019-2022_Leverne/PFT07"
    # GPP GC
    gpp_post_sif_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_POST_SIF/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Lev"
    gpp_post_gpp_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_POST_GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07"
    gpp_post_sif_gpp_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_POST_SIF-GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Lev"
    gpp_post_sif_gpp_bacour_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_POST_SIF-GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Bac"
    gpp_prior_path = "/home/surface9/vtartagl/Pixel_homogenes/Simus/PRIOR_GPP_2000-2022/PFT07"
    # IN SITU GPP
    fluxnet_prior_path ="/home/surface9/vtartagl/assimilation_TROPOMI-SIF+is-GPP/PRIOR_PFT07_Lev/"
    fluxnet_post_sif_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_FLUXNET_POST_SIF/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Lev/"
    fluxnet_post_gpp_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_FLUXNET_POST_GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07/"
    fluxnet_post_sif_gpp_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_FLUXNET_POST_SIF-GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Lev"
    fluxnet_post_sif_gpp_bacour_path = "/home/surface9/vtartagl/EVAL_POST/ARTICLE_LEVERNE/GPP_FLUXNET_POST_SIF-GPP/kF0.1_no_ASJ-ASV-EJMAX-LMAX_bounds_LACRIT_GDD-TH-1.5/PFT07_Bac"

    # --- 3. PROCESS DATA ---
    sif_eval, sif_eval_rmsd = load_eval_sif_pixels(
        eval_post_sif_path, eval_post_gpp_path, eval_post_sif_gpp_path,
        eval_post_sif_gpp_bacour_path, prior_sif_path,
        pixel_filter_list=EVALUATION_PIXELS
    )

    gpp_rmsd = {'prior_fluxcom': [], 'prior_fluxsat': [], 'post_sif_fluxcom': [], 'post_sif_fluxsat': [], 'post_gpp_fluxcom': [], 'post_gpp_fluxsat': [], 'post_sif_gpp_fluxcom': [], 'post_sif_gpp_fluxsat': [], 'post_sif_gpp_bacour_fluxcom': [], 'post_sif_gpp_bacour_fluxsat': []}
    
    print(f"\nProcessing GPP pixels. Applying filter for {len(EVALUATION_PIXELS)} pixels.")
    for px_num in EVALUATION_PIXELS: 
        px = str(px_num)
        files = {
            'post_sif': get_most_recent_file(os.path.join(gpp_post_sif_path, f'PX{px}', 'output_*.nc')),
            'post_gpp': get_most_recent_file(os.path.join(gpp_post_gpp_path, f'PX{px}', 'output_*.nc')),
            'post_sif_gpp': get_most_recent_file(os.path.join(gpp_post_sif_gpp_path, f'PX{px}', 'output_*.nc')),
            'post_sif_gpp_bacour': get_most_recent_file(os.path.join(gpp_post_sif_gpp_bacour_path, f'PX{px}', 'output_*.nc')),
            'prior': get_most_recent_file(os.path.join(gpp_prior_path, f'PX{px}', 'output_*.nc')),
            'fluxcom': get_most_recent_file(f'/home/surface9/vtartagl/Pixel_homogenes/GPP_estimations/FLUXCOM_pixels/PFT07/GPP_FLUXCOM_720x1440_daily_pft07_px{px}_*.nc'),
            'fluxsat': get_most_recent_file(f'/home/surface9/vtartagl/Pixel_homogenes/GPP_estimations/FLUXSAT_pixels/0.25_remapcon/PFT07/GPP_FluxSat_720x1440_daily_pft07_px{px}_*.nc')
        }
        if not all(files.values()): continue

        gpp_data_sif = load_gpp_data(files['prior'], files['post_sif'], files['fluxcom'], files['fluxsat'])
        gpp_data_gpp = load_gpp_data(files['prior'], files['post_gpp'], files['fluxcom'], files['fluxsat'])
        gpp_data_sif_gpp = load_gpp_data(files['prior'], files['post_sif_gpp'], files['fluxcom'], files['fluxsat'])
        gpp_data_sif_gpp_bacour = load_gpp_data(files['prior'], files['post_sif_gpp_bacour'], files['fluxcom'], files['fluxsat'])

        if not all([gpp_data_sif, gpp_data_gpp, gpp_data_sif_gpp, gpp_data_sif_gpp_bacour]): continue
        
        gpp_rmsd['prior_fluxcom'].append(calculate_rmsd(gpp_data_sif['fluxcom'], gpp_data_sif['prior']))
        gpp_rmsd['post_sif_fluxcom'].append(calculate_rmsd(gpp_data_sif['fluxcom'], gpp_data_sif['post']))
        gpp_rmsd['post_gpp_fluxcom'].append(calculate_rmsd(gpp_data_gpp['fluxcom'], gpp_data_gpp['post']))
        gpp_rmsd['post_sif_gpp_fluxcom'].append(calculate_rmsd(gpp_data_sif_gpp['fluxcom'], gpp_data_sif_gpp['post']))
        gpp_rmsd['post_sif_gpp_bacour_fluxcom'].append(calculate_rmsd(gpp_data_sif_gpp_bacour['fluxcom'], gpp_data_sif_gpp_bacour['post']))
        
        gpp_rmsd['prior_fluxsat'].append(calculate_rmsd(gpp_data_sif['fluxsat'], gpp_data_sif['prior']))
        gpp_rmsd['post_sif_fluxsat'].append(calculate_rmsd(gpp_data_sif['fluxsat'], gpp_data_sif['post']))
        gpp_rmsd['post_gpp_fluxsat'].append(calculate_rmsd(gpp_data_gpp['fluxsat'], gpp_data_gpp['post']))
        gpp_rmsd['post_sif_gpp_fluxsat'].append(calculate_rmsd(gpp_data_sif_gpp['fluxsat'], gpp_data_sif_gpp['post']))
        gpp_rmsd['post_sif_gpp_bacour_fluxsat'].append(calculate_rmsd(gpp_data_sif_gpp_bacour['fluxsat'], gpp_data_sif_gpp_bacour['post']))

    fluxnet_gpp_rmsd, fluxnet_gpp_data = process_fluxnet_gpp(
        fluxnet_prior_path, fluxnet_post_sif_path, fluxnet_post_gpp_path,
        fluxnet_post_sif_gpp_path, fluxnet_post_sif_gpp_bacour_path,
        site_filter_list=EVALUATION_SITES
    )

    # --- 4. CHECK AND PLOT ---
    sif_cell_count = len(sif_eval_rmsd.get('prior', []))
    gpp_cell_count = len(gpp_rmsd.get('prior_fluxcom', []))
    gpp_site_count = len(fluxnet_gpp_rmsd.get('prior', []))

    print(f"\nFinal Check:")
    print(f"  SIF Cells Processed:   {sif_cell_count} / {len(EVALUATION_PIXELS)}")
    print(f"  GPP Cells Processed:   {gpp_cell_count} / {len(EVALUATION_PIXELS)}")
    print(f"  GPP Sites Processed:   {gpp_site_count} / {len(EVALUATION_SITES)}")

    if sif_cell_count > 0 and gpp_cell_count > 0 and gpp_site_count > 0:
        print("\nAll data processed successfully. Generating plots...")
        plot_rmsd_boxplots(gpp_rmsd=gpp_rmsd,
                           fluxnet_gpp_rmsd=fluxnet_gpp_rmsd,
                           sif_eval_rmsd=sif_eval_rmsd,
                           sif_cell_count=sif_cell_count,
                           gpp_site_count=gpp_site_count,
                           gpp_cell_count=gpp_cell_count)
    else:
        print("\nCould not generate plots: No valid data was found for one or more filtered categories.")